# Pipeline Experimentations

Notebook structure:
- Cell 1-5: setup and reusable helpers
- Cell 6-9: instrumented pipeline functions and sweep definitions
- Cell 10-12: orchestrator/CVAE sweeps and pipeline reference run
- Cell 13-14: target length challenge (2..12), figures, cleanup

This notebook is pipeline-only: no non-agentic baseline comparison is executed.

In [21]:
import os
import sys
import gc
import math
import json
import random
import time
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / 'database' / 'training_data.json'
EXPERIMENT_DIR = REPO_ROOT / 'experiments' / 'pipeline_experimentations'
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

RUN_PROFILE = 'full'  # 'fast' or 'full'
RUN_ORCHESTRATOR_SWEEP = True
RUN_CVAE_SWEEP = True
RUN_LENGTH_SWEEP = True
USE_ESM_BIOLOGIST = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
try:
    torch.random.manual_seed(SEED)
except Exception:
    pass

print(f'Repo root: {REPO_ROOT}')
print(f'Data path exists: {DATA_PATH.exists()}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Run profile: {RUN_PROFILE}')

Repo root: /home/arthur/Documents/M2/pfe/multi-expert-peptide-pipeline
Data path exists: True
CUDA available: True
Run profile: full


In [22]:
from peptide_pipeline.dataloader.dataloader_json import DataLoader as JSONDataLoader
from peptide_pipeline.generator.cvae_generator import CVAEGenerator
from peptide_pipeline.chemist.agent_v1.config_chemist import ChemistConfig, RangeTarget
from peptide_pipeline.chemist.agent_v1.chemist_agent import ChemistAgent
from peptide_pipeline.orchestrator.orchestrator import Orchestrator
from peptide_pipeline.biologist.base import BaseBiologist

TARGET = {
    'dbaasp_id': 'DBAASPS_373',
    'sequence': 'KLFKRWKHLFR',
    'length': 11,
    'ph': None,
    'molecular_weight': 1558.9480000000003,
    'logp': -0.992100000000006,
    'net_charge': 5.0,
    'isoelectric_point': 12.18,
    'hydrophobicity': 1.05,
    'cathionicity': 6,
}

AA = 'ACDEFGHIKLMNPQRSTVWY'
AA_TO_IDX = {aa: i for i, aa in enumerate(AA)}
PAD_IDX = 20
VOCAB_SIZE = 21
MAX_LEN = 14

class FallbackBiologist(BaseBiologist):
    def __init__(self, reference_peptide: str):
        self.reference_peptide = reference_peptide

    def score_peptides(self, peptides: List[str]) -> List[float]:
        if not peptides:
            return []
        ref_set = set(self.reference_peptide)
        denom = max(len(ref_set), 1)
        return [len(ref_set.intersection(set(p))) / denom for p in peptides]

    def predict_activity(self, peptides: List[str], context: Any = None) -> List[float]:
        if isinstance(context, str) and context.strip():
            previous = self.reference_peptide
            self.reference_peptide = context
            scores = self.score_peptides(peptides)
            self.reference_peptide = previous
            return scores
        return self.score_peptides(peptides)


def build_biologist(reference_peptide: str, score_temperature: float = 50.0):
    if USE_ESM_BIOLOGIST:
        try:
            from peptide_pipeline.biologist.esm_biologist_global_l2 import ESMBiologistGlobalL2
            return ESMBiologistGlobalL2(
                reference_peptide=reference_peptide,
                batch_size=16,
                score_temperature=score_temperature,
            )
        except Exception as e:
            print(f'ESM biologist unavailable, using fallback biologist: {e}')
    return FallbackBiologist(reference_peptide=reference_peptide)

In [23]:
def encode_one_hot_with_pad(sequences: List[str], max_len: int = MAX_LEN) -> torch.Tensor:
    x = torch.zeros(len(sequences), max_len * VOCAB_SIZE, dtype=torch.float32)
    for i, seq in enumerate(sequences):
        for pos in range(max_len):
            x[i, pos * VOCAB_SIZE + PAD_IDX] = 1.0
        for pos, aa in enumerate(seq[:max_len]):
            if aa in AA_TO_IDX:
                x[i, pos * VOCAB_SIZE + PAD_IDX] = 0.0
                x[i, pos * VOCAB_SIZE + AA_TO_IDX[aa]] = 1.0
    return x


def build_condition_tensor(dataframe: pd.DataFrame, condition_dim: int = 32) -> torch.Tensor:
    cond = torch.zeros(len(dataframe), condition_dim, dtype=torch.float32)
    cond[:, 0] = torch.tensor(dataframe['length'].values, dtype=torch.float32)
    cond[:, 1] = torch.tensor(dataframe['molecular_weight'].values, dtype=torch.float32)
    cond[:, 2] = torch.tensor(dataframe['net_charge'].values, dtype=torch.float32)
    cond[:, 3] = torch.tensor(dataframe['isoelectric_point'].values, dtype=torch.float32)
    cond[:, 4] = torch.tensor(dataframe['hydrophobicity'].values, dtype=torch.float32)
    cond[:, 5] = torch.tensor(dataframe['cathionicity'].values, dtype=torch.float32)
    cond[:, 6] = 0.5
    cond[:, 7] = torch.tensor(dataframe['logp'].values, dtype=torch.float32)
    cond[:, 8] = 0.0
    cond[:, 9] = 5.0
    cond[:, 10] = 5.0
    cond[:, 11] = 100.0
    return cond


loader = JSONDataLoader()
loader.load_data(
    source=str(DATA_PATH),
    columns=[
        'sequence', 'length', 'ph', 'molecular_weight', 'logp',
        'net_charge', 'isoelectric_point', 'hydrophobicity', 'cathionicity'
    ],
    fillna_defaults={
        'length': 10,
        'ph': 7.0,
        'molecular_weight': 1500.0,
        'logp': 0.0,
        'net_charge': 0.0,
        'isoelectric_point': 7.0,
        'hydrophobicity': 0.0,
        'cathionicity': 0.0,
    },
    normalize_sequence=True,
    sequence_column='sequence',
    keep_standard_amino_acids_only=True,
)

df = loader.get_data().copy()
sequences = df['sequence'].tolist()
lengths = torch.tensor(df['length'].astype(int).values, dtype=torch.long)
x = encode_one_hot_with_pad(sequences, max_len=MAX_LEN)
conditions = build_condition_tensor(df, condition_dim=32)

print(f'Dataset rows: {len(df)}')
print(f'x shape: {tuple(x.shape)}')
print(f'conditions shape: {tuple(conditions.shape)}')
display(df.head(3))

Dataset rows: 4410
x shape: (4410, 294)
conditions shape: (4410, 32)


,sequence,length,ph,molecular_weight,logp,net_charge,isoelectric_point,hydrophobicity,cathionicity
0,KVVVKWVVKVVK,12,7.0,1648.291,5.6026,5.0,14.0,-1.07,4
1,LFIFFF,6,7.0,832.059,3.2860,1.0,14.0,-3.25,0
2,KAAAKWAAKAAK,12,7.0,1451.913,1.1499,5.0,14.0,0.33,4


In [24]:
def build_chemist_from_target(
    target: Dict[str, Any],
    width_scale: float = 1.0,
    length_min: float = 8.0,
    length_max: float = 14.0,
    length_weight: float = 1.0,
) -> ChemistAgent:
    ph = 7.0 if target['ph'] is None else float(target['ph'])
    target_length = float(target['length'])
    min_len = min(length_min, target_length)
    max_len = max(length_max, target_length)

    target_mw = float(target['molecular_weight'])
    # Strictness scaling: allow broader range if target is small/large?
    # Keeping original logic for now
    mw_span = max(250.0, 0.35 * target_mw)
    mw_min = max(200.0, target_mw - mw_span)
    mw_max = target_mw + mw_span

    target_charge = float(target['net_charge'])
    charge_span = max(1.5, 0.5 * abs(target_charge))
    charge_min = max(0.0, target_charge - charge_span)
    charge_max = target_charge + charge_span

    target_ip = float(target['isoelectric_point'])
    ip_span = 2.5 * width_scale
    ip_min = max(3.0, target_ip - ip_span)
    ip_max = min(14.0, target_ip + ip_span)

    target_hydro = float(target['hydrophobicity'])
    hydro_span = 1.5 * width_scale
    hydro_min = target_hydro - hydro_span
    hydro_max = target_hydro + hydro_span

    target_logp = float(target['logp'])
    logp_span = 2.0 * width_scale
    logp_min = target_logp - logp_span
    logp_max = target_logp + logp_span

    return ChemistAgent(
        config=ChemistConfig(
            ph=ph,
            length=RangeTarget(min=min_len, max=max_len, target=target_length, weight=length_weight),
            molecular_weight=RangeTarget(min=mw_min, max=mw_max, target=target_mw, weight=1.0),
            logp=RangeTarget(min=logp_min, max=logp_max, target=target_logp, weight=1.0),
            net_charge=RangeTarget(min=charge_min, max=charge_max, target=target_charge, weight=1.0),
            isoelectric_point=RangeTarget(min=ip_min, max=ip_max, target=target_ip, weight=1.0),
            hydrophobicity=RangeTarget(min=hydro_min, max=hydro_max, target=target_hydro, weight=1.0),
        )
    )

def model_cache_path(
    latent_dim: int,
    hidden_dim: int,
    epochs: int,
    lr: float,
    kl_anneal_epochs: int = 0,
    batch_size: int = 64,
) -> Path:
    lr_text = str(lr).replace('.', 'p')
    return EXPERIMENT_DIR / (
        f'cvae_lat{latent_dim}_hid{hidden_dim}_ep{epochs}_lr{lr_text}'
        f'_kl{kl_anneal_epochs}_bs{batch_size}.pth'
    )


def get_or_train_cvae(
    latent_dim: int,
    hidden_dim: int,
    epochs: int,
    lr: float,
    kl_anneal_epochs: int,
    batch_size: int = 64,
) -> CVAEGenerator:
    model = CVAEGenerator(max_len=MAX_LEN, latent_dim=latent_dim, hidden_dim=hidden_dim, condition_dim=32)
    path = model_cache_path(
        latent_dim=latent_dim,
        hidden_dim=hidden_dim,
        epochs=epochs,
        lr=lr,
        kl_anneal_epochs=kl_anneal_epochs,
        batch_size=batch_size,
    )

    if path.exists():
        try:
            model.load_model(str(path))
            print(f"Loaded cached model: {path.name}")
            return model
        except Exception as e:
            print(f"Failed to load cached model {path.name}, retraining. Error: {e}")

    print(f"Training CVAE Model: {path.name} ...")
    device = model.device
    x_local = x.to(device)
    c_local = conditions.to(device)
    l_local = lengths.to(device)

    model.train_model(
        data=x_local,
        conditions=c_local,
        lengths=l_local,
        epochs=epochs,
        batch_size=batch_size,
        lr=lr,
        kl_anneal_epochs=kl_anneal_epochs,
    )
    model.save_model(str(path))
    print(f"Model saved to {path.name}")
    return model

In [25]:
def normalized_hamming(a: str, b: str) -> float:
    m = max(len(a), len(b), 1)
    a_pad = a.ljust(m, '-')
    b_pad = b.ljust(m, '-')
    return sum(ch1 != ch2 for ch1, ch2 in zip(a_pad, b_pad)) / m


def bigram_entropy(peptides: List[str]) -> float:
    counts: Dict[str, int] = {}
    total = 0
    for p in peptides:
        for i in range(max(len(p) - 1, 0)):
            bg = p[i : i + 2]
            counts[bg] = counts.get(bg, 0) + 1
            total += 1
    if total == 0:
        return 0.0
    probs = np.array([v / total for v in counts.values()], dtype=float)
    return float(-(probs * np.log2(np.clip(probs, 1e-12, 1.0))).sum())


def summarize_topk(rows: List[Dict[str, Any]]) -> Dict[str, float]:
    if not rows:
        return {
            'top1_score': 0.0,
            'topk_mean_score': 0.0,
            'validity_rate': 0.0,
            'unique_ratio': 0.0,
            'mean_pairwise_distance': 0.0,
            'bigram_entropy': 0.0,
        }

    peptides = [r['peptide'] for r in rows]
    scores = [float(r.get('combined_score', r.get('score', 0.0))) for r in rows]
    in_limits = [bool(r.get('in_limits', False)) for r in rows]

    dists = []
    for i in range(len(peptides)):
        for j in range(i + 1, len(peptides)):
            dists.append(normalized_hamming(peptides[i], peptides[j]))

    return {
        'top1_score': float(scores[0]),
        'topk_mean_score': float(np.mean(scores)),
        'validity_rate': float(np.mean(in_limits)),
        'unique_ratio': float(len(set(peptides)) / max(len(peptides), 1)),
        'mean_pairwise_distance': float(np.mean(dists) if dists else 0.0),
        'bigram_entropy': bigram_entropy(peptides),
    }

In [26]:
def safe_float(value: Any, default: float = 0.0) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def run_instrumented_pipeline(
    generator: CVAEGenerator,
    chemist: ChemistAgent,
    biologist: BaseBiologist,
    final_target: Dict[str, Any],
    nb_iterations: int,
    nb_peptides: int,
    top_k: int,
    exploration_rate: float,
    random_parent_count: int,
) -> Tuple[List[Dict[str, Any]], pd.DataFrame]:
    orchestrator = Orchestrator(generator=generator, chemist=chemist, biologist=biologist)
    target_constraints = {
        'length': final_target['length'],
        'molecular_weight': final_target['molecular_weight'],
        'logp': final_target['logp'],
        'net_charge': final_target['net_charge'],
        'isoelectric_point': final_target['isoelectric_point'],
        'hydrophobicity': final_target['hydrophobicity'],
        'cathionicity': final_target['cathionicity'],
    }

    top_rows = orchestrator.run(
        nb_iterations=nb_iterations,
        nb_peptides=nb_peptides,
        top_k=top_k,
        exploration_rate=exploration_rate,
        initial_peptide=final_target['sequence'],
        final_target=target_constraints,
        random_parent_count=random_parent_count,
    )

    # Reconstruct compact iteration trace from returned rows.
    trace_rows: List[Dict[str, Any]] = []
    if top_rows:
        by_iter = pd.DataFrame(top_rows).groupby('iteration', as_index=False)
        for _, chunk in by_iter:
            trace_rows.append(
                {
                    'iteration': int(chunk['iteration'].iloc[0]),
                    'returned_count': int(len(chunk)),
                    'best_combined_score': float(chunk['combined_score'].max()),
                    'mean_combined_score': float(chunk['combined_score'].mean()),
                    'validity_rate_returned': float(chunk['in_limits'].astype(float).mean()),
                }
            )

    return top_rows, pd.DataFrame(trace_rows).sort_values('iteration').reset_index(drop=True)


def run_non_agentic_baseline(
    generator: CVAEGenerator,
    chemist: ChemistAgent,
    biologist: BaseBiologist,
    final_target: Dict[str, Any],
    budget: int,
    top_k: int,
) -> List[Dict[str, Any]]:
    constraints = {
        'length': final_target['length'],
        'molecular_weight': final_target['molecular_weight'],
        'logp': final_target['logp'],
        'net_charge': final_target['net_charge'],
        'isoelectric_point': final_target['isoelectric_point'],
        'hydrophobicity': final_target['hydrophobicity'],
        'cathionicity': final_target['cathionicity'],
    }

    peptides = generator.generate_peptides(count=budget, constraints=constraints)
    chem_rows = chemist.evaluate_peptides(peptides)
    valid_rows = [r for r in chem_rows if r.get('sequence')]
    bio_scores = biologist.score_peptides([r['sequence'] for r in valid_rows])

    rows = []
    for c_row, b_score in zip(valid_rows, bio_scores):
        chem_score = safe_float(c_row.get('score', 0.0))
        bio_score = safe_float(b_score)
        combined = 0.5 * (chem_score + bio_score)
        rows.append(
            {
                'peptide': c_row['sequence'],
                'score': combined,
                'combined_score': combined,
                'chemist_score': chem_score,
                'biologist_score': bio_score,
                'in_limits': bool(c_row.get('in_limits', False)),
                'properties': c_row.get('properties', {}),
                'iteration': 1,
            }
        )

    rows = sorted(
        rows,
        key=lambda x: (x['in_limits'], x['combined_score'], x['biologist_score'], x['chemist_score']),
        reverse=True,
    )
    dedup = {}
    for row in rows:
        if row['peptide'] not in dedup:
            dedup[row['peptide']] = row
    return list(dedup.values())[:top_k]

In [27]:
# Definition of Baseline and Sweeps
# We establish a strong baseline, then vary parameters one by one to isolate effects.

BASELINE_ORCHESTRATOR = {
    'nb_iterations': 20,
    'nb_peptides': 128,
    'top_k': 10,
    'exploration_rate': 0.1,
    'random_parent_count': 4,
}  # A balanced starting point

BASELINE_CVAE = {
    'latent_dim': 64,
    'hidden_dim': 256,
    'epochs': 120,
    'lr': 0.001,
    'kl_anneal_epochs': 40,
    'batch_size': 64,
}

if RUN_PROFILE == 'fast':
    # Expanded but still practical for notebook runs.
    ORCHESTRATOR_VARIANTS = {
        'nb_iterations': [10, 30, 40],
        'nb_peptides': [64, 192, 320],
        'top_k': [5, 20, 32],
        'exploration_rate': [0.0, 0.25, 0.5, 0.75],
        'random_parent_count': [0, 8, 16, 32],
    }

    CVAE_VARIANTS = {
        'latent_dim': [32, 96, 128],
        'hidden_dim': [128, 384, 512],
        'epochs': [80, 160, 240],
        'lr': [0.0005, 0.0007, 0.0015],
        'kl_anneal_epochs': [10, 20, 80, 120],
        'batch_size': [32, 128],
    }

    REPEATS = 2
else:
    # Wider exploration for report-quality runs.
    ORCHESTRATOR_VARIANTS = {
        'nb_iterations': [5, 10, 30, 50, 75],
        'nb_peptides': [64, 128, 256, 384, 512],
        'top_k': [5, 10, 20, 32, 50],
        'exploration_rate': [0.0, 0.1, 0.2, 0.4, 0.6, 0.8],
        'random_parent_count': [0, 4, 10, 20, 32],
    }

    CVAE_VARIANTS = {
        'latent_dim': [32, 64, 96, 128, 160],
        'hidden_dim': [128, 256, 384, 512],
        'epochs': [60, 120, 200, 280],
        'lr': [0.0003, 0.0005, 0.0007, 0.001, 0.0015],
        'kl_anneal_epochs': [5, 10, 20, 40, 80, 120],
        'batch_size': [32, 64, 128],
    }

    REPEATS = 10  # Rigorous statistical representation (10+ runs per point).

# Curated multi-parameter variants complement one-at-a-time analysis.
ORCHESTRATOR_COMBO_VARIANTS = [
    {
        'name': 'combo_quality_focus',
        'overrides': {
            'nb_iterations': 40,
            'nb_peptides': 192,
            'top_k': 20,
            'exploration_rate': 0.2,
            'random_parent_count': 8,
        },
    },
    {
        'name': 'combo_throughput_focus',
        'overrides': {
            'nb_iterations': 12,
            'nb_peptides': 384,
            'top_k': 12,
            'exploration_rate': 0.05,
            'random_parent_count': 0,
        },
    },
    {
        'name': 'combo_exploration_focus',
        'overrides': {
            'nb_iterations': 30,
            'nb_peptides': 128,
            'top_k': 16,
            'exploration_rate': 0.7,
            'random_parent_count': 20,
        },
    },
]

CVAE_COMBO_VARIANTS = [
    {
        'name': 'combo_compact_fast',
        'overrides': {
            'latent_dim': 32,
            'hidden_dim': 128,
            'epochs': 100,
            'lr': 0.0015,
            'kl_anneal_epochs': 20,
            'batch_size': 128,
        },
    },
    {
        'name': 'combo_capacity_focus',
        'overrides': {
            'latent_dim': 128,
            'hidden_dim': 512,
            'epochs': 220,
            'lr': 0.0007,
            'kl_anneal_epochs': 80,
            'batch_size': 64,
        },
    },
    {
        'name': 'combo_regularized',
        'overrides': {
            'latent_dim': 96,
            'hidden_dim': 384,
            'epochs': 180,
            'lr': 0.0005,
            'kl_anneal_epochs': 120,
            'batch_size': 32,
        },
    },
]

print(f"Run Profile: {RUN_PROFILE}")
print(f"Repeats per experiment: {REPEATS}")
print(f"Baseline Orchestrator: {BASELINE_ORCHESTRATOR}")
print(f"Baseline CVAE: {BASELINE_CVAE}")
print(f"Orchestrator one-at-a-time variants: {sum(len(v) for v in ORCHESTRATOR_VARIANTS.values())}")
print(f"CVAE one-at-a-time variants: {sum(len(v) for v in CVAE_VARIANTS.values())}")
print(f"Orchestrator combo variants: {len(ORCHESTRATOR_COMBO_VARIANTS)}")
print(f"CVAE combo variants: {len(CVAE_COMBO_VARIANTS)}")

Run Profile: full
Repeats per experiment: 10
Baseline Orchestrator: {'nb_iterations': 20, 'nb_peptides': 128, 'top_k': 10, 'exploration_rate': 0.1, 'random_parent_count': 4}
Baseline CVAE: {'latent_dim': 64, 'hidden_dim': 256, 'epochs': 120, 'lr': 0.001, 'kl_anneal_epochs': 40, 'batch_size': 64}
Orchestrator one-at-a-time variants: 26
CVAE one-at-a-time variants: 27
Orchestrator combo variants: 3
CVAE combo variants: 3


In [ ]:
orchestrator_records = []
orchestrator_traces = []

if RUN_ORCHESTRATOR_SWEEP:
    # 1. Train the reference CVAE model (Baseline config) once.
    print("Training CVAE Baseline for Orchestrator Sweep...")
    fixed_model = get_or_train_cvae(**BASELINE_CVAE)

    # 2. Reuse a single biologist across all orchestrator configs.
    biologist = build_biologist(TARGET['sequence'], score_temperature=50.0)

    # 3. Prepare experiment list (Baseline + One-At-A-Time + Combo Variants).
    experiment_tasks = []

    # Baseline task
    experiment_tasks.append({
        'name': 'Baseline',
        'params': BASELINE_ORCHESTRATOR.copy(),
        'param_name': 'baseline',
        'param_value': 'baseline',
        'sweep_group': 'baseline',
    })

    # One-at-a-time tasks
    for param_key, variants in ORCHESTRATOR_VARIANTS.items():
        for val in variants:
            cfg = BASELINE_ORCHESTRATOR.copy()
            cfg[param_key] = val
            experiment_tasks.append({
                'name': f"{param_key}={val}",
                'params': cfg,
                'param_name': param_key,
                'param_value': val,
                'sweep_group': 'one_at_a_time',
            })

    # Curated combo tasks
    for combo in ORCHESTRATOR_COMBO_VARIANTS:
        cfg = BASELINE_ORCHESTRATOR.copy()
        cfg.update(combo['overrides'])
        experiment_tasks.append({
            'name': combo['name'],
            'params': cfg,
            'param_name': 'combo_profile',
            'param_value': combo['name'],
            'sweep_group': 'combo',
        })

    print(f"Total Orchestrator Configurations to run: {len(experiment_tasks)}")

    # 4. Execute tasks
    for i, task in enumerate(experiment_tasks, start=1):
        run_cfg = task['params']
        param_name = task['param_name']
        param_val = task['param_value']
        sweep_group = task['sweep_group']

        per_cfg_metrics = []
        per_cfg_times = []
        total_generated = int(run_cfg['nb_iterations'] * run_cfg['nb_peptides'])

        print(f"Processing config {i}/{len(experiment_tasks)}: {task['name']} (Repeats: {REPEATS})")

        for rep in range(1, REPEATS + 1):
            # Seed strategy: base seed + config index + repeat index.
            seed_val = SEED + 10_000 + (100 * i) + rep
            random.seed(seed_val)
            np.random.seed(seed_val)
            try:
                torch.random.manual_seed(seed_val)
            except Exception:
                pass

            chemist = build_chemist_from_target(TARGET)

            run_start = time.perf_counter()
            top_rows, trace_df = run_instrumented_pipeline(
                generator=fixed_model,
                chemist=chemist,
                biologist=biologist,
                final_target=TARGET,
                nb_iterations=run_cfg['nb_iterations'],
                nb_peptides=run_cfg['nb_peptides'],
                top_k=run_cfg['top_k'],
                exploration_rate=run_cfg['exploration_rate'],
                random_parent_count=run_cfg['random_parent_count'],
            )
            elapsed_sec = float(time.perf_counter() - run_start)

            metrics = summarize_topk(top_rows)
            per_cfg_metrics.append(metrics)
            per_cfg_times.append(elapsed_sec)

            orchestrator_records.append(
                {
                    'config_name': task['name'],
                    'sweep_group': sweep_group,
                    'param_name': param_name,
                    'param_value': param_val,
                    'repeat': rep,
                    'total_generated': total_generated,
                    'elapsed_sec': elapsed_sec,
                    **run_cfg,
                    **metrics,
                }
            )

            if not trace_df.empty:
                trace_df = trace_df.copy()
                trace_df['config_name'] = task['name']
                trace_df['sweep_group'] = sweep_group
                trace_df['repeat'] = rep
                trace_df['elapsed_sec'] = elapsed_sec
                orchestrator_traces.append(trace_df)

        top1_vals = [m['top1_score'] for m in per_cfg_metrics]
        mean_top1 = np.mean(top1_vals)
        std_top1 = np.std(top1_vals, ddof=1) if len(top1_vals) > 1 else 0.0
        mean_time = np.mean(per_cfg_times) if per_cfg_times else 0.0
        std_time = np.std(per_cfg_times, ddof=1) if len(per_cfg_times) > 1 else 0.0
        print(f"  -> Finished: top1={mean_top1:.4f} +/- {std_top1:.4f} | time={mean_time:.2f}s +/- {std_time:.2f}s")

orchestrator_raw_df = pd.DataFrame(orchestrator_records)
orchestrator_trace_df = pd.concat(orchestrator_traces, ignore_index=True) if orchestrator_traces else pd.DataFrame()
orchestrator_results_df = pd.DataFrame()

if not orchestrator_raw_df.empty:
    orchestrator_results_df = orchestrator_raw_df.groupby(
        ['config_name', 'sweep_group', 'param_name', 'param_value'],
        as_index=False,
    ).agg({
        'top1_score': ['mean', 'std'],
        'topk_mean_score': ['mean', 'std'],
        'validity_rate': ['mean', 'std'],
        'unique_ratio': ['mean', 'std'],
        'mean_pairwise_distance': ['mean', 'std'],
        'elapsed_sec': ['mean', 'std'],
        'total_generated': 'first',
    })
    orchestrator_results_df.columns = ['_'.join(c).strip('_') for c in orchestrator_results_df.columns]
    orchestrator_results_df = orchestrator_results_df.sort_values(
        ['top1_score_mean', 'validity_rate_mean'],
        ascending=[False, False],
    ).reset_index(drop=True)

display(orchestrator_results_df)
display(orchestrator_raw_df.head(10))

Training CVAE Baseline for Orchestrator Sweep...
Training CVAE Model: cvae_lat64_hid256_ep120_lr0p001_kl40_bs64.pth ...
  Epoch 050/120 | Recon: 1.4768 | KL: 0.0564 | KL weight: 1.00


In [ ]:
cvae_records = []

if RUN_CVAE_SWEEP:
    # Use baseline orchestrator settings for CVAE comparison.
    orch_config = BASELINE_ORCHESTRATOR
    biologist = build_biologist(TARGET['sequence'], score_temperature=50.0)

    # 1. Prepare experiment list (Baseline + One-At-A-Time + Combo Variants).
    cvae_tasks = []

    # Baseline task
    cvae_tasks.append({
        'name': 'Baseline',
        'params': BASELINE_CVAE.copy(),
        'param_name': 'baseline',
        'param_value': 'baseline',
        'sweep_group': 'baseline',
    })

    # One-at-a-time tasks
    for param_key, variants in CVAE_VARIANTS.items():
        for val in variants:
            cfg = BASELINE_CVAE.copy()
            cfg[param_key] = val
            cvae_tasks.append({
                'name': f"{param_key}={val}",
                'params': cfg,
                'param_name': param_key,
                'param_value': val,
                'sweep_group': 'one_at_a_time',
            })

    # Curated combo tasks
    for combo in CVAE_COMBO_VARIANTS:
        cfg = BASELINE_CVAE.copy()
        cfg.update(combo['overrides'])
        cvae_tasks.append({
            'name': combo['name'],
            'params': cfg,
            'param_name': 'combo_profile',
            'param_value': combo['name'],
            'sweep_group': 'combo',
        })

    print(f"Total CVAE Configurations to run: {len(cvae_tasks)}")

    # 2. Execute CVAE tasks
    for i, task in enumerate(cvae_tasks, start=1):
        cvae_cfg = task['params']
        param_name = task['param_name']
        param_value = task['param_value']
        sweep_group = task['sweep_group']
        print(f"Processing CVAE config {i}/{len(cvae_tasks)}: {task['name']} (Repeats: {REPEATS})")

        per_cfg_metrics = []
        per_cfg_times = []

        for rep in range(1, REPEATS + 1):
            seed_val = SEED + 20_000 + (100 * i) + rep
            random.seed(seed_val)
            np.random.seed(seed_val)
            try:
                torch.random.manual_seed(seed_val)
            except Exception:
                pass

            # Train/load model for this specific config (not timed inside run).
            model = get_or_train_cvae(**cvae_cfg)
            chemist = build_chemist_from_target(TARGET)

            run_start = time.perf_counter()
            top_rows, trace_df = run_instrumented_pipeline(
                generator=model,
                chemist=chemist,
                biologist=biologist,
                final_target=TARGET,
                nb_iterations=orch_config['nb_iterations'],
                nb_peptides=orch_config['nb_peptides'],
                top_k=orch_config['top_k'],
                exploration_rate=orch_config['exploration_rate'],
                random_parent_count=orch_config['random_parent_count'],
            )
            elapsed_sec = float(time.perf_counter() - run_start)

            metrics = summarize_topk(top_rows)
            per_cfg_metrics.append(metrics)
            per_cfg_times.append(elapsed_sec)

            cvae_records.append({
                'config_name': task['name'],
                'sweep_group': sweep_group,
                'param_name': param_name,
                'param_value': param_value,
                'repeat': rep,
                'elapsed_sec': elapsed_sec,
                **cvae_cfg,
                **metrics,
            })

            print(f"  > Rep {rep}: top1={metrics['top1_score']:.3f} val_rate={metrics['validity_rate']:.2f} time={elapsed_sec:.2f}s")

        top1_vals = [m['top1_score'] for m in per_cfg_metrics]
        mean_top1 = np.mean(top1_vals)
        std_top1 = np.std(top1_vals, ddof=1) if len(top1_vals) > 1 else 0.0
        mean_time = np.mean(per_cfg_times) if per_cfg_times else 0.0
        std_time = np.std(per_cfg_times, ddof=1) if len(per_cfg_times) > 1 else 0.0
        print(f"  -> Finished {task['name']}: top1={mean_top1:.4f} +/- {std_top1:.4f} | time={mean_time:.2f}s +/- {std_time:.2f}s")

cvae_raw_df = pd.DataFrame(cvae_records)
cvae_results_df = pd.DataFrame()

if not cvae_raw_df.empty:
    cvae_results_df = cvae_raw_df.groupby(
        ['config_name', 'sweep_group', 'param_name', 'param_value'],
        as_index=False,
    ).agg({
        'top1_score': ['mean', 'std'],
        'topk_mean_score': ['mean', 'std'],
        'validity_rate': ['mean', 'std'],
        'unique_ratio': ['mean', 'std'],
        'mean_pairwise_distance': ['mean', 'std'],
        'elapsed_sec': ['mean', 'std'],
        'latent_dim': 'first',
        'hidden_dim': 'first',
        'epochs': 'first',
        'lr': 'first',
        'kl_anneal_epochs': 'first',
        'batch_size': 'first',
    })
    cvae_results_df.columns = ['_'.join(c).strip('_') for c in cvae_results_df.columns]
    cvae_results_df = cvae_results_df.sort_values(
        ['top1_score_mean', 'validity_rate_mean', 'mean_pairwise_distance_mean'],
        ascending=[False, False, False],
    ).reset_index(drop=True)

display(cvae_results_df)
display(cvae_raw_df.head(10))

In [ ]:
pipeline_reference_rows = []

# Pipeline-only reference run using the best CVAE found in the sweep.
if 'cvae_results_df' in globals() and not cvae_results_df.empty:
    best_row = cvae_results_df.iloc[0]
    best_cvae_cfg = {
        'latent_dim': int(best_row['latent_dim_first']),
        'hidden_dim': int(best_row['hidden_dim_first']),
        'epochs': int(best_row['epochs_first']),
        'lr': float(best_row.get('lr_first', BASELINE_CVAE['lr'])),
        'kl_anneal_epochs': int(best_row.get('kl_anneal_epochs_first', BASELINE_CVAE['kl_anneal_epochs'])),
        'batch_size': int(best_row.get('batch_size_first', BASELINE_CVAE['batch_size'])),
    }
    print(f"Using best CVAE config from sweep: {best_cvae_cfg}")
else:
    best_cvae_cfg = BASELINE_CVAE
    print(f"No CVAE sweep results found. Using baseline CVAE: {best_cvae_cfg}")

chosen_model = get_or_train_cvae(**best_cvae_cfg)

base_orch = BASELINE_ORCHESTRATOR
biologist = build_biologist(TARGET['sequence'], score_temperature=50.0)

print(f"Running pipeline reference repeats: {REPEATS}")

for rep in range(1, REPEATS + 1):
    random.seed(SEED + 30_000 + rep)
    np.random.seed(SEED + 30_000 + rep)
    try:
        torch.random.manual_seed(SEED + 30_000 + rep)
    except Exception:
        pass

    chemist = build_chemist_from_target(TARGET)

    run_start = time.perf_counter()
    pipe_rows, _ = run_instrumented_pipeline(
        generator=chosen_model,
        chemist=chemist,
        biologist=biologist,
        final_target=TARGET,
        nb_iterations=base_orch['nb_iterations'],
        nb_peptides=base_orch['nb_peptides'],
        top_k=base_orch['top_k'],
        exploration_rate=base_orch['exploration_rate'],
        random_parent_count=base_orch['random_parent_count'],
    )
    elapsed_sec = float(time.perf_counter() - run_start)

    pipe_metrics = summarize_topk(pipe_rows)
    pipeline_reference_rows.append({'repeat': rep, 'method': 'pipeline_reference', 'elapsed_sec': elapsed_sec, **pipe_metrics})

pipeline_reference_df = pd.DataFrame(pipeline_reference_rows)
pipeline_reference_summary_df = pd.DataFrame()
if not pipeline_reference_df.empty:
    pipeline_reference_summary_df = pipeline_reference_df.groupby('method', as_index=False).agg({
        'top1_score': ['mean', 'std'],
        'topk_mean_score': ['mean', 'std'],
        'validity_rate': ['mean', 'std'],
        'unique_ratio': ['mean', 'std'],
        'mean_pairwise_distance': ['mean', 'std'],
        'bigram_entropy': ['mean', 'std'],
        'elapsed_sec': ['mean', 'std'],
    })
    pipeline_reference_summary_df.columns = ['_'.join(col).strip('_') for col in pipeline_reference_summary_df.columns.values]

display(pipeline_reference_summary_df)
display(pipeline_reference_df.head(10))

In [ ]:
# Length-compression study: can we keep quality with shorter target peptides?
length_sweep_rows = []

if RUN_LENGTH_SWEEP:
    # Use the baseline CVAE model for a controlled length sweep.
    chosen_model = get_or_train_cvae(**BASELINE_CVAE)

    base_run = BASELINE_ORCHESTRATOR

    # Target lengths: 2 to 12 as requested.
    length_candidates = list(range(2, 13))

    biologist_len = build_biologist(TARGET['sequence'], score_temperature=50.0)
    original_len = float(TARGET['length'])

    print(f"Running length sweep on lengths: {length_candidates}")

    for target_len in length_candidates:
        ratio = float(target_len) / max(original_len, 1.0)
        compressed_target = dict(TARGET)
        compressed_target['length'] = int(target_len)

        # Scale length-dependent target properties so short targets remain feasible.
        compressed_target['molecular_weight'] = float(TARGET['molecular_weight']) * ratio
        compressed_target['net_charge'] = max(1.0, float(TARGET['net_charge']) * ratio)

        width_scale = 1.0 + (0.5 if target_len < 5 else (0.2 if target_len < TARGET['length'] else 0.0))
        min_allowed = max(2.0, float(target_len) - 1.0)
        max_allowed = min(float(MAX_LEN), float(target_len) + 1.0)

        chemist_len = build_chemist_from_target(
            compressed_target,
            width_scale=width_scale,
            length_min=min_allowed,
            length_max=max_allowed,
            length_weight=1.5,
        )

        for rep in range(1, REPEATS + 1):
            random.seed(SEED + 40_000 + (100 * target_len) + rep)
            np.random.seed(SEED + 40_000 + (100 * target_len) + rep)
            try:
                torch.random.manual_seed(SEED + 40_000 + (100 * target_len) + rep)
            except Exception:
                pass

            run_start = time.perf_counter()
            top_rows, _ = run_instrumented_pipeline(
                generator=chosen_model,
                chemist=chemist_len,
                biologist=biologist_len,
                final_target=compressed_target,
                nb_iterations=base_run['nb_iterations'],
                nb_peptides=base_run['nb_peptides'],
                top_k=base_run['top_k'],
                exploration_rate=base_run['exploration_rate'],
                random_parent_count=base_run['random_parent_count'],
            )
            elapsed_sec = float(time.perf_counter() - run_start)

            metrics = summarize_topk(top_rows)
            realized_lengths = [
                float(r.get('properties', {}).get('length', len(r.get('peptide', ''))))
                for r in top_rows
            ]
            mean_realized_len = float(np.mean(realized_lengths)) if realized_lengths else 0.0

            length_sweep_rows.append(
                {
                    'target_length': int(target_len),
                    'repeat': rep,
                    'elapsed_sec': elapsed_sec,
                    'mean_realized_length': mean_realized_len,
                    'top1_score': metrics['top1_score'],
                    'topk_mean_score': metrics['topk_mean_score'],
                    'validity_rate': metrics['validity_rate'],
                    'mean_pairwise_distance': metrics['mean_pairwise_distance'],
                }
            )

        current_rows = [r for r in length_sweep_rows if r['target_length'] == target_len]
        avg_top1 = np.mean([r['top1_score'] for r in current_rows])
        avg_time = np.mean([r['elapsed_sec'] for r in current_rows])
        print(f"  Length {target_len}: top1={avg_top1:.3f} | time={avg_time:.2f}s")

length_sweep_df = pd.DataFrame(length_sweep_rows)
length_sweep_summary_df = pd.DataFrame()

if not length_sweep_df.empty:
    length_sweep_summary_df = length_sweep_df.groupby('target_length', as_index=False).agg({
        'mean_realized_length': ['mean', 'std'],
        'top1_score': ['mean', 'std'],
        'topk_mean_score': ['mean', 'std'],
        'validity_rate': ['mean', 'std'],
        'mean_pairwise_distance': ['mean', 'std'],
        'elapsed_sec': ['mean', 'std'],
    })
    length_sweep_summary_df.columns = ['_'.join(c).strip('_') for c in length_sweep_summary_df.columns]

    base_row = length_sweep_summary_df[length_sweep_summary_df['target_length'] == int(TARGET['length'])]
    if not base_row.empty:
        base_top1 = float(base_row['top1_score_mean'].iloc[0])
        length_sweep_summary_df['top1_retention_vs_original'] = length_sweep_summary_df['top1_score_mean'] / max(base_top1, 1e-9)
    else:
        length_sweep_summary_df['top1_retention_vs_original'] = np.nan

display(length_sweep_df.head(20))
display(length_sweep_summary_df)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    print("Seaborn not found, falling back to matplotlib defaults.")
    plt.style.use('ggplot')


def align_dual_y_ticks(ax, ax2, n_ticks=6):
    # Force both y-axes to use the same number of major ticks.
    ax.yaxis.set_major_locator(MaxNLocator(nbins=n_ticks))
    ax2.yaxis.set_major_locator(MaxNLocator(nbins=n_ticks))

    # Recompute left ticks and map them to the right axis range so horizontal levels align.
    y1_ticks = ax.get_yticks()
    y1_min, y1_max = ax.get_ylim()
    y2_min, y2_max = ax2.get_ylim()

    if len(y1_ticks) > 1 and y1_max > y1_min:
        y2_ticks = np.interp(y1_ticks, [y1_min, y1_max], [y2_min, y2_max])

        # Keep alignment, but hide lowest/highest right-axis ticks for cleaner rendering.
        if len(y2_ticks) > 2:
            ax2.set_yticks(y2_ticks[1:-1])
        else:
            ax2.set_yticks(y2_ticks)


def style_dual_axis(ax, ax2):
    # Keep one shared grid from the left axis and force it behind the data.
    ax.grid(True, which='major', axis='both', alpha=0.22, zorder=0)
    ax.set_axisbelow(True)
    ax2.grid(False)

    # Reduce overlap between left/right y labels and ticks.
    ax.yaxis.labelpad = 8
    ax2.yaxis.labelpad = 12
    ax2.spines['right'].set_position(('outward', 10))

    # Align left/right horizontal levels.
    align_dual_y_ticks(ax, ax2, n_ticks=6)


def plot_single_parameter_sweep(df, param_name, ax=None):
    if df.empty:
        return
    subset = df[df['param_name'] == param_name].sort_values('param_value')
    if subset.empty:
        return

    try:
        subset['param_value'] = pd.to_numeric(subset['param_value'])
        subset = subset.sort_values('param_value')
    except Exception:
        pass

    if ax is None:
        _, ax = plt.subplots(figsize=(7.4, 4.8))

    ax2 = ax.twinx()

    # First line: quality
    h1 = ax.errorbar(
        subset['param_value'],
        subset['top1_score_mean'],
        yerr=subset['top1_score_std'].fillna(0.0),
        fmt='-o',
        capsize=4,
        color='tab:blue',
        linewidth=2.0,
        markersize=5,
        zorder=4,
        label='Top-1 score (mean ± std)',
    )
    ax.set_xlabel(param_name)
    ax.set_ylabel('Top-1 score', color='tab:blue')
    ax.tick_params(axis='y', labelcolor='tab:blue')

    # Second line: runtime
    h2 = ax2.errorbar(
        subset['param_value'],
        subset['elapsed_sec_mean'],
        yerr=subset['elapsed_sec_std'].fillna(0.0),
        fmt='--s',
        capsize=4,
        color='tab:orange',
        linewidth=1.9,
        markersize=5,
        zorder=5,
        label='Runtime per run (s, mean ± std)',
    )
    ax2.set_ylabel('Runtime per run (seconds)', color='tab:orange')
    ax2.tick_params(axis='y', labelcolor='tab:orange')

    style_dual_axis(ax, ax2)

    # Combined legend
    ax.legend([h1, h2], ['Top-1 score (mean ± std)', 'Runtime per run (s, mean ± std)'], loc='best')


# 1) Orchestrator one-at-a-time sensitivity with runtime cost
if 'orchestrator_results_df' in globals() and not orchestrator_results_df.empty:
    params = [p for p in orchestrator_results_df['param_name'].unique() if p != 'baseline']

    n_cols = 2
    n_rows = max(1, (len(params) + n_cols - 1) // n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(13.6, 4.8 * n_rows), constrained_layout=True)
    axes = np.array(axes).reshape(-1)

    for i, p in enumerate(params):
        plot_single_parameter_sweep(orchestrator_results_df, p, axes[i])
        axes[i].set_title(f'Orchestrator: effect of {p}')

    for j in range(len(params), len(axes)):
        axes[j].axis('off')

    fig.suptitle('Orchestrator Hyperparameter Sensitivity (Score + Runtime)', y=1.02, fontsize=14)
    plt.show()


# 2) CVAE one-at-a-time sensitivity with runtime cost
if 'cvae_results_df' in globals() and not cvae_results_df.empty:
    params = [p for p in cvae_results_df['param_name'].unique() if p != 'baseline']

    n_cols = 2
    n_rows = max(1, (len(params) + n_cols - 1) // n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(13.6, 4.8 * n_rows), constrained_layout=True)
    axes = np.array(axes).reshape(-1)

    for i, p in enumerate(params):
        plot_single_parameter_sweep(cvae_results_df, p, axes[i])
        axes[i].set_title(f'CVAE: effect of {p}')

    for j in range(len(params), len(axes)):
        axes[j].axis('off')

    fig.suptitle('CVAE Hyperparameter Sensitivity (Score + Runtime)', y=1.02, fontsize=14)
    plt.show()


# 3) Target length challenge (2..12): score + runtime
if 'length_sweep_summary_df' in globals() and not length_sweep_summary_df.empty:
    fig, ax = plt.subplots(figsize=(9.0, 5.4), constrained_layout=True)
    metrics = length_sweep_summary_df.sort_values('target_length').copy()

    ax2 = ax.twinx()

    h1 = ax.errorbar(
        metrics['target_length'],
        metrics['top1_score_mean'],
        yerr=metrics['top1_score_std'].fillna(0.0),
        fmt='-o',
        capsize=4,
        color='tab:blue',
        linewidth=2.0,
        markersize=5,
        zorder=4,
        label='Top-1 score (mean ± std)',
    )
    ax.set_xlabel('Target length (residues)')
    ax.set_ylabel('Top-1 score', color='tab:blue')
    ax.tick_params(axis='y', labelcolor='tab:blue')
    ax.set_title('Pipeline Performance vs Target Peptide Length (Score + Runtime)')

    h2 = ax2.errorbar(
        metrics['target_length'],
        metrics['elapsed_sec_mean'],
        yerr=metrics['elapsed_sec_std'].fillna(0.0),
        fmt='--s',
        capsize=4,
        color='tab:orange',
        linewidth=1.9,
        markersize=5,
        zorder=5,
        label='Runtime per run (s, mean ± std)',
    )
    ax2.set_ylabel('Runtime per run (seconds)', color='tab:orange')
    ax2.tick_params(axis='y', labelcolor='tab:orange')

    style_dual_axis(ax, ax2)
    ax.legend([h1, h2], ['Top-1 score (mean ± std)', 'Runtime per run (s, mean ± std)'], loc='best')

    plt.show()


# 4) Pipeline-only reference summary
if 'pipeline_reference_summary_df' in globals() and not pipeline_reference_summary_df.empty:
    display(pipeline_reference_summary_df)

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Cleanup done.')